In [86]:
# Using -q for a quieter installation in the notebook
!pip install datasets tiktoken tqdm numpy matplotlib pandas -q

# --- Core Torch and Helpers ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import time
from contextlib import nullcontext
from dataclasses import dataclass
from torch import tensor

# --- Data Handling ---
import numpy as np
from datasets import load_dataset
import tiktoken
from tqdm.auto import tqdm

# --- Plotting and Analysis ---
import matplotlib.pyplot as plt
import pandas as pd

print("All dependencies installed and imported successfully.")
# Verify GPU is available
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected. Running on CPU.")


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
All dependencies installed and imported successfully.
GPU detected: NVIDIA GeForce RTX 5090
PyTorch version: 2.8.0.dev20250319+cu128
CUDA version: 12.8


In [87]:
# --- Global Training Config ---
BATCH_SIZE = 32          # Number of sequences in a batch
BLOCK_SIZE = 256         # Context length of the model
MAX_ITERS = 5000         # Total training iterations
EVAL_INTERVAL = 250      # How often to evaluate and print loss
EVAL_ITERS = 100         # Number of batches to average for validation loss
LEARNING_RATE = 3e-4     # Max learning rate for the optimizer
WARMUP_ITERS = 200       # Number of iterations for learning rate warmup
MIN_LR = 3e-5            # Minimum learning rate after cosine decay

In [88]:
# --- Device and Precision Config ---
# Automatically selects 'cuda' if available, otherwise 'cpu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Use bfloat16 for training if available (on A100+ GPUs), otherwise float16.
# This maximizes performance and prevents common float16 overflow issues.
DTYPE = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
PTDTYPE = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}[DTYPE]
# Create the autocast context manager for mixed-precision training
CTX = nullcontext() if DEVICE == 'cpu' else torch.amp.autocast(device_type=DEVICE, dtype=PTDTYPE)
# GradScaler is only needed for float16, not bfloat16 or float32
SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))

/tmp/ipykernel_102/3088237724.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))


In [ ]:
# --- Base Model Architecture Config ---
# We will use these settings for all four models to ensure a fair comparison.
@dataclass
class BaseModelConfig:
    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 256
    vocab_size: int = 50257  # Standard for GPT-2 tokenizer
    block_size: int = BLOCK_SIZE
    dropout: float = 0.1
    bias: bool = False # Set to False for LayerNorms and Linears, as is common practice
    
    MLA_COMPRESSION_RATIO = 0.5
    d_latent: int = int(256 * MLA_COMPRESSION_RATIO) # d_model * ratio
    d_rope: int = 16 

In [90]:
# Create an instance of the config
BASE_CONFIG = BaseModelConfig()

In [91]:
GQA_N_GROUPS = 2
assert BASE_CONFIG.n_head % GQA_N_GROUPS == 0, "n_head must be divisible by n_groups for GQA"

In [92]:
print(f"Running on device: {DEVICE}")
print(f"Using data type: {DTYPE}")
print(f"Base model config: {BASE_CONFIG.n_layer} layers, {BASE_CONFIG.n_head} heads, {BASE_CONFIG.n_embd} embedding dim.")

Running on device: cuda
Using data type: bfloat16
Base model config: 6 layers, 8 heads, 256 embedding dim.


In [93]:
# --- Dataset Configuration ---
DATASET_NAME = "roneneldan/TinyStories"
TRAIN_BIN_FILE = "train.bin"
VAL_BIN_FILE = "val.bin"
TOKENIZER_MODEL = "gpt2"

In [94]:
# --- Tokenization and File Writing ---
if not (os.path.exists(TRAIN_BIN_FILE) and os.path.exists(VAL_BIN_FILE)):
    print("Binary data files not found. Starting data download and tokenization...")

    # Load the dataset from Hugging Face
    ds = load_dataset(DATASET_NAME)

    # Initialize the tokenizer
    enc = tiktoken.get_encoding(TOKENIZER_MODEL)
    assert enc.max_token_value < 2**16, "Tokenizer vocab size is too large for uint16"

    def process_and_tokenize(example):
        """Tokenize a single text example and return token IDs and length."""
        ids = enc.encode_ordinary(example['text'])
        return {'ids': ids, 'len': len(ids)}

    # Tokenize the dataset in parallel
    tokenized_ds = ds.map(
        process_and_tokenize,
        remove_columns=['text'],
        desc="Tokenizing splits",
        num_proc=os.cpu_count(),
    )

    # Write tokenized data to memory-mapped binary files
    for split, dset in tokenized_ds.items():
        # Use a consistent naming convention (validation -> val)
        split_name = 'val' if split == 'validation' else split
        filename = f"{split_name}.bin"

        total_tokens = np.sum(dset['len'], dtype=np.uint64)
        print(f"Found {total_tokens:,} tokens in the '{split}' split.")

        # Use uint16 for memory efficiency, as vocab size allows it
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(total_tokens,))

        print(f"Writing tokens to {filename}...")
        current_idx = 0
        # Process in batches to manage memory during concatenation
        for batch in tqdm(dset.iter(batch_size=2048), total=len(dset)//2048):
            concatenated_ids = np.concatenate(batch['ids'])
            arr[current_idx : current_idx + len(concatenated_ids)] = concatenated_ids
            current_idx += len(concatenated_ids)

        arr.flush()
    print("Tokenization and file writing complete.")
else:
    print("Found existing train.bin and val.bin files. Skipping data preparation.")

Found existing train.bin and val.bin files. Skipping data preparation.


In [ ]:
# --- Data Loading Function for Training ---
def get_batch(split):
    """
    Loads a batch of data from the appropriate binary file.
    This function is designed to be called repeatedly during training.
    """
    filename = TRAIN_BIN_FILE if split == 'train' else VAL_BIN_FILE
    # Re-map the file in each call to avoid potential memory issues with long-running processes
    data = np.memmap(filename, dtype=np.uint16, mode='r')

    # Generate random starting indices for each sequence in the batch
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))

    # Create input (x) and target (y) sequences
    x = torch.stack([torch.from_numpy((data[i:i+BLOCK_SIZE]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+BLOCK_SIZE]).astype(np.int64)) for i in ix])

    # Move data to the configured device
    return x.to(DEVICE), y.to(DEVICE)

print("\nData loading utilities are ready.")


Data loading utilities are ready.


In [96]:
class LayerNorm(nn.Module):
    """ A custom LayerNorm implementation. """
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

In [97]:
class MLP(nn.Module):
    """ The standard Feed-Forward Network (FFN) block. """
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

In [98]:
@torch.no_grad()
def estimate_loss(model, ctx):
    """
    Evaluates the model's loss on the train and validation sets.
    """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            with ctx:
                # The forward pass now only returns what we need
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("Shared `estimate_loss` function is defined.")

Shared `estimate_loss` function is defined.


In [99]:
class MultiHeadAttention(nn.Module):
    """
    A standard Multi-Head Attention module with an explicit KV Cache for efficient inference.
    This implementation uses a single linear layer to project Q, K, and V simultaneously
    for efficiency, and leverages PyTorch's built-in scaled_dot_product_attention.
    """
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        # A single linear layer to project inputs to Q, K, V simultaneously
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Output projection layer
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # Regularization
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, kv_cache=None):
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality
        head_size = C // self.n_head

        # 1. Project to Q, K, V for all heads in batch and split
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # 2. Reshape Q, K, V to prepare for multi-head attention
        # (B, T, C) -> (B, T, n_head, head_size) -> (B, n_head, T, head_size)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)

        # --- KV Cache Logic ---
        if kv_cache is not None:
            # This is an inference step (T=1). Concatenate with the cached keys and values.
            cached_k, cached_v = kv_cache
            # Concatenate along the sequence length dimension (dim=2)
            k = torch.cat([cached_k, k], dim=2)
            v = torch.cat([cached_v, v], dim=2)
        
        # The current keys and values become the new cache for the next step.
        current_kv_cache = (k, v)

        # 3. Perform attention using PyTorch's efficient implementation.
        # is_causal=True handles masking for training/prefill where T > 1.
        # During inference with a KV cache, T=1, so is_causal=False is correct as we attend to all past tokens.
        y = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=(kv_cache is None) # Causal mask only for training/prefill (T > 1)
        )

        # 4. Concatenate heads and project back to the embedding dimension
        # (B, n_head, T, head_size) -> (B, T, n_head, head_size) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        
        return y, current_kv_cache

print("Attention Mechanism 1: Multi-Head Attention (MHA) with KV Cache defined.")

Attention Mechanism 1: Multi-Head Attention (MHA) with KV Cache defined.
